In [0]:
%run ../config_setup

In [0]:
# Databricks Notebook: audit_end (parameterized — env config_setup se)
from delta.tables import DeltaTable
from pyspark.sql.functions import current_timestamp, expr
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

# Workflow mein task ka jo naam hai, woh yahan daalo (jaise "audit_start")
TASK_NAME = "audit_start" 

try:
    run_id = dbutils.jobs.taskValues.get(taskKey=TASK_NAME, key="run_id")
    
    # 👉 env-aware: main.audit nahi — config_setup ka AUDIT
    delta_table = DeltaTable.forName(spark, f"{AUDIT}.job_audit_log")
    
    delta_table.update(
        condition = f"run_id = '{run_id}'",
        set = {
            "end_ts": current_timestamp(),
            "status": expr("'COMPLETED'"),
            "is_completed": expr("'Y'"),
            "execution_time_sec": expr("round(unix_timestamp(current_timestamp()) - unix_timestamp(start_ts), 2)")
        }
    )
    print(f"✅ Audit END updated for Run ID: {run_id}")

except Exception as e:
    print(f"❌ Error: {e}")